# Binary Logistic Regression

In [1]:
# Binary Classification Demo: Logistic Regression (Mock dataset)
# - Pure Python + scikit-learn
# - Prints accuracy + confusion matrix + learned weights

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ----------------------------
# 1) Mock (synthetic) dataset
# ----------------------------
# We generate 2 numeric features:
#   x1 = "study_hours"
#   x2 = "sleep_hours"
# Label rule (with noise):
#   more study + decent sleep -> higher chance to pass (y=1)
rng = np.random.default_rng(42)
N = 400

study_hours = rng.uniform(0, 10, size=N)   # 0..10
sleep_hours = rng.uniform(3, 9, size=N)    # 3..9

# Linear score + noise
noise = rng.normal(0, 1.2, size=N)
score = 1.1 * study_hours + 0.7 * sleep_hours - 8.0 + noise

# Convert score -> probability via sigmoid, then sample label
prob = 1.0 / (1.0 + np.exp(-score))
y = (rng.uniform(0, 1, size=N) < prob).astype(int)

# Feature matrix
X = np.column_stack([study_hours, sleep_hours])

print("Dataset shape:", X.shape, "Labels (0/1) counts:", np.bincount(y))

# ----------------------------
# 2) Train / Test split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

# ----------------------------
# 3) Train Logistic Regression
# ----------------------------
# solver='liblinear' is great for small binary problems
model = LogisticRegression()
model.fit(X_train, y_train)

# ----------------------------
# 4) Evaluate
# ----------------------------
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\n=== Test Results ===")
print("Accuracy:", round(acc, 4))
print("Confusion matrix [[TN FP],[FN TP]]:\n", cm)
print("\nClassification report:\n", classification_report(y_test, y_pred, digits=4))

# ----------------------------
# 5) Show learned parameters
# ----------------------------
print("Intercept (b):", model.intercept_[0])
print("Weights (w1, w2):", model.coef_[0])
print("Feature order: [study_hours, sleep_hours]")

# ----------------------------
# 6) Predict a few new samples
# ----------------------------
new_X = np.array([
    [2.0, 4.0],   # low study, low sleep
    [8.0, 6.0],   # high study, ok sleep
    [5.0, 8.0],   # mid study, high sleep
], dtype=float)

pred_class = model.predict(new_X)
pred_prob = model.predict_proba(new_X)[:, 1]

print("\n=== New samples ===")
for i, (x, c, p) in enumerate(zip(new_X, pred_class, pred_prob), start=1):
    print(f"Sample {i}: study={x[0]:.1f}, sleep={x[1]:.1f} -> class={c}, P(y=1)={p:.3f}")


Dataset shape: (400, 2) Labels (0/1) counts: [142 258]

=== Test Results ===
Accuracy: 0.89
Confusion matrix [[TN FP],[FN TP]]:
 [[28  7]
 [ 4 61]]

Classification report:
               precision    recall  f1-score   support

           0     0.8750    0.8000    0.8358        35
           1     0.8971    0.9385    0.9173        65

    accuracy                         0.8900       100
   macro avg     0.8860    0.8692    0.8766       100
weighted avg     0.8893    0.8900    0.8888       100

Intercept (b): -5.873737079819338
Weights (w1, w2): [0.76603786 0.5501515 ]
Feature order: [study_hours, sleep_hours]

=== New samples ===
Sample 1: study=2.0, sleep=4.0 -> class=0, P(y=1)=0.105
Sample 2: study=8.0, sleep=6.0 -> class=1, P(y=1)=0.972
Sample 3: study=5.0, sleep=8.0 -> class=1, P(y=1)=0.914
